# 🎓 DegreeDetailExtract — Donut Fine-Tuning Notebook

**Pipeline**: Synthetic certificate generation → CORD smoke test → Donut fine-tuning → Evaluation → Inference

**Model**: `naver-clova-ix/donut-base` (Swin-B encoder + BART-style decoder)

**GPU target**: Colab Free Tier (T4, ~15 GB VRAM)

---
Run all sections in order. Each section header explains what it does.


## 🛠️ Section 0 — Environment Setup


In [ ]:
# Install system fonts so the Pillow certificate generator has TrueType fonts
!apt-get install -y -q fonts-liberation2 fonts-dejavu-core

# Install Python packages
!pip install -q \
    'transformers>=4.35.0' \
    'datasets>=2.14.0' \
    'accelerate>=0.24.0' \
    sentencepiece \
    Faker \
    'Pillow>=10.0.0' \
    'albumentations>=1.3.0' \
    tqdm \
    'scikit-learn>=1.3.0' \
    editdistance \
    matplotlib

print('\n✅ All packages installed.')

In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'CUDA ver: {torch.version.cuda}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {mem_gb:.1f} GB')
else:
    print('⚠️  No GPU detected — training will be very slow on CPU.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice  : {DEVICE}')

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PATH   = '/content/drive/MyDrive/DegreeDetailExtract'
DATASET_PATH = f'{DRIVE_PATH}/dataset'
CKPT_PATH    = f'{DRIVE_PATH}/checkpoints'

for d in [DATASET_PATH, CKPT_PATH]:
    os.makedirs(d, exist_ok=True)

print(f'Drive root : {DRIVE_PATH}')
print(f'Dataset    : {DATASET_PATH}')
print(f'Checkpoints: {CKPT_PATH}')

## 🖼️ Section 1 — Synthetic Data Generation

**Before running this section**, make sure the project files are in `/content/DegreeDetailExtract/`.

**Option A** (recommended): Clone from GitHub
```
!git clone https://github.com/YOUR_USERNAME/DegreeDetailExtract.git /content/DegreeDetailExtract
```

**Option B**: Use the Colab left-sidebar file browser to upload `generate_certificates.py` and the `generator/` folder to `/content/DegreeDetailExtract/`.


In [ ]:
import os, sys

REPO_DIR = '/content/DegreeDetailExtract'

# ── Option A: uncomment and set your GitHub URL ────────────────────────────────
# REPO_URL = 'https://github.com/YOUR_USERNAME/DegreeDetailExtract.git'
# if not os.path.exists(REPO_DIR):
#     !git clone {REPO_URL} {REPO_DIR}

# ── Verify files exist ─────────────────────────────────────────────────────────
assert os.path.exists(f'{REPO_DIR}/generate_certificates.py'), \
    f'❌ generate_certificates.py not found in {REPO_DIR}. Upload or clone first.'
assert os.path.exists(f'{REPO_DIR}/generator/templates.py'), \
    f'❌ generator/templates.py not found. Upload the generator/ folder.'

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%cd {REPO_DIR}
print(f'✅ Project files verified in {REPO_DIR}')

In [ ]:
# ── Generate 5 000 synthetic certificates ─────────────────────────────────────
# Estimated time: ~15–25 min on Colab CPU (Pillow rendering is CPU-bound)
# Use --no-augment --count 15 for a quick template preview

!python generate_certificates.py \
    --count 5000 \
    --output_dir /content/dataset \
    --seed 42 \
    --quality 90

print('\nGeneration complete.')

In [ ]:
import json, random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

IMG_DIR = Path('/content/dataset/images')
files   = sorted(IMG_DIR.glob('*.jpg'))
print(f'Total images: {len(files)}')

# Show 4 random samples
samples = random.sample(files, min(4, len(files)))
fig, axes = plt.subplots(1, 4, figsize=(22, 10))
for ax, fpath in zip(axes, samples):
    ax.imshow(Image.open(fpath))
    ax.set_title(fpath.name, fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Generated Certificates', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Verify metadata
with open('/content/dataset/metadata_train.jsonl') as f:
    records = [json.loads(l) for l in f]
print(f'\nTrain records: {len(records)}')
print('Sample record:')
print(json.dumps(records[0], indent=2))

In [ ]:
import shutil, os

# Compress the dataset into a single zip file.
# Copying 1 file to Drive is ~20x faster than copying 5,000 individual images.
print("Compressing dataset (1-2 min) ...")
zip_src = '/content/dataset_archive'
shutil.make_archive(zip_src, 'zip', '/content', 'dataset')
zip_gb  = os.path.getsize(zip_src + '.zip') / 1e9
print(f"Zip created: {zip_gb:.2f} GB")

# Copy the single zip to Drive
dest_zip = f'{DRIVE_PATH}/dataset_archive.zip'
print(f"Uploading to Drive: {dest_zip} ...")
shutil.copy(zip_src + '.zip', dest_zip)
print("Dataset zip saved to Drive.")

# Also copy the small metadata jsonl files separately (handy for quick access)
for split in ('train', 'val', 'test'):
    src = f'/content/dataset/metadata_{split}.jsonl'
    dst = f'{DRIVE_PATH}/metadata_{split}.jsonl'
    shutil.copy(src, dst)
    n = sum(1 for _ in open(src))
    print(f"  {split}: {n:,} records saved.")

## 🔥 Section 2 — CORD Pipeline Smoke Test

Trains Donut on CORD (structured receipt data) for ~200 steps to verify the
training loop, tokenisation, and generation all work correctly before
touching the certificate data.

> Expected loss after 200 steps: should clearly decrease from the initial value.


In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch

print('Loading DonutProcessor and model from naver-clova-ix/donut-base ...')
processor = DonutProcessor.from_pretrained('naver-clova-ix/donut-base')
model     = VisionEncoderDecoderModel.from_pretrained('naver-clova-ix/donut-base')
model.to(DEVICE)
print(f'✅ Model loaded  ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)')

In [ ]:
import json
from torch.utils.data import Dataset
from datasets import load_dataset as hf_load

CORD_TASK_START = '<s_cord-v2>'
CORD_TASK_END   = '</s_cord-v2>'
MAX_LENGTH      = 512

# Add CORD task token if not already present
if CORD_TASK_START not in processor.tokenizer.all_special_tokens:
    processor.tokenizer.add_special_tokens({'additional_special_tokens': [CORD_TASK_START, CORD_TASK_END]})
    model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(CORD_TASK_START)
model.config.pad_token_id           = processor.tokenizer.pad_token_id


class CordDataset(Dataset):
    def __init__(self, dataset):
        self.data = dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image  = sample['image'].convert('RGB')
        gt     = json.loads(sample['ground_truth'])
        target = f"{CORD_TASK_START}{json.dumps(gt)}{CORD_TASK_END}"

        pixel_values = processor(image, return_tensors='pt').pixel_values.squeeze()
        input_ids    = processor.tokenizer(
            target,
            add_special_tokens=False,
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        ).input_ids.squeeze()

        labels = input_ids.clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}


print('Loading CORD dataset ...')
cord_raw = hf_load('naver-clova-ix/cord-v2')
cord_train = CordDataset(cord_raw['train'])
cord_val   = CordDataset(cord_raw['validation'])
print(f'CORD train: {len(cord_train)}  val: {len(cord_val)}')

In [ ]:
import torch
import os
from torch.utils.data import DataLoader
from torch.optim import AdamW

# ── Memory: set allocator config BEFORE any CUDA allocation ───────────────────
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

# Gradient checkpointing: trades compute for memory (essential on T4 + Donut)
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

CORD_STEPS = 200
LR         = 5e-5

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'labels':       torch.stack([b['labels']       for b in batch]),
    }

# batch_size=1: Donut's Swin encoder is large; bs=2 OOMs the T4's 15 GB
loader    = DataLoader(cord_train, batch_size=1, shuffle=True, collate_fn=collate_fn)
optimizer = AdamW(model.parameters(), lr=LR)

# torch.amp (updated API — torch.cuda.amp.* is deprecated in PyTorch >= 2.3)
use_amp = (DEVICE == 'cuda')
scaler  = torch.amp.GradScaler('cuda', enabled=use_amp)

model.train()
step, losses = 0, []

for batch in loader:
    pixel_values = batch['pixel_values'].to(DEVICE)
    labels       = batch['labels'].to(DEVICE)

    with torch.amp.autocast('cuda', enabled=use_amp):
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)  # set_to_none frees grad buffers immediately

    losses.append(loss.item())
    step += 1
    if step % 50 == 0:
        avg = sum(losses[-50:]) / len(losses[-50:])
        print(f'Step {step:4d} / {CORD_STEPS} | loss: {avg:.4f}')
    if step >= CORD_STEPS:
        break

print(f'\nCORD smoke test complete. Final loss: {losses[-1]:.4f}')
print('Loss should have decreased from the initial value -- if so, the pipeline is working.')

In [ ]:
# Quick inference on one CORD validation sample
model.eval()
sample = cord_raw['validation'][0]
image  = sample['image'].convert('RGB')
gt     = sample['ground_truth']

pixel_values = processor(image, return_tensors='pt').pixel_values.to(DEVICE)
decoder_input_ids = torch.full(
    (1, 1),
    model.config.decoder_start_token_id,
    device=DEVICE
)

with torch.no_grad():
    outputs = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=MAX_LENGTH,
        early_stopping=True,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )

pred = processor.tokenizer.decode(outputs.sequences[0], skip_special_tokens=False)
print('=== Ground truth (first 300 chars) ===')
print(gt[:300])
print('\n=== Model prediction (first 300 chars) ===')
print(pred[:300])

## 📚 Section 3 — Donut Fine-Tuning on Synthetic Certificates

**Prerequisites**: Dataset must be in Google Drive at `DATASET_PATH`.  
Run Sections 0 and 1 first, or symlink an existing dataset.

**T4 hyperparameter choices**:
- Batch 2 × grad-accum 8 = **effective batch 16**
- fp16 mixed precision
- Checkpoints saved every 500 steps to Drive (survives Colab disconnects)


In [ ]:
import gc, torch
from transformers import DonutProcessor, VisionEncoderDecoderModel

# Free the CORD smoke-test model from GPU memory before loading a fresh one.
# Without this, both models compete for the T4's 15 GB and cause OOM.
try:
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print("Cleared previous model from VRAM.")
except NameError:
    pass  # first run -- nothing to clear

# Reload a clean base model for certificate fine-tuning
print("Loading fresh donut-base ...")
processor = DonutProcessor.from_pretrained('naver-clova-ix/donut-base')
model     = VisionEncoderDecoderModel.from_pretrained('naver-clova-ix/donut-base')

# Special tokens for the 7 certificate fields
CERT_SPECIAL_TOKENS = [
    '<s_cert>', '</s_cert>',
    '<s_student_name>',    '</s_student_name>',
    '<s_university_name>', '</s_university_name>',
    '<s_course_name>',     '</s_course_name>',
    '<s_specialization>',  '</s_specialization>',
    '<s_pass_class>',      '</s_pass_class>',
    '<s_authority_name>',  '</s_authority_name>',
    '<s_issue_date>',      '</s_issue_date>',
]

processor.tokenizer.add_special_tokens(
    {'additional_special_tokens': CERT_SPECIAL_TOKENS}
)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids('<s_cert>')
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.eos_token_id

model.to(DEVICE)
print(f"Special tokens added ({len(CERT_SPECIAL_TOKENS)} new tokens).")
print(f"Tokenizer vocab size: {len(processor.tokenizer)}")

In [ ]:
import json, os, shutil
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset

# Unzip dataset from Drive to local Colab SSD if not already present.
# Reading images from local SSD during training is ~10x faster than reading
# from the Drive FUSE mount.
LOCAL_DATASET = '/content/dataset'
if not os.path.exists(LOCAL_DATASET):
    print("Unzipping dataset from Drive to local storage ...")
    shutil.unpack_archive(f'{DRIVE_PATH}/dataset_archive.zip', '/content')
    print("Done.")
else:
    print("Dataset already available locally -- skipping unzip.")

DATASET_PATH = LOCAL_DATASET   # override to local path for training

CERT_MAX_LENGTH = 512
FIELD_NAMES = [
    'student_name', 'university_name', 'course_name',
    'specialization', 'pass_class', 'authority_name', 'issue_date',
]


def fields_to_target(fields: dict) -> str:
    inner = ''.join(
        f'<s_{f}>{fields.get(f, "")}</s_{f}>'
        for f in FIELD_NAMES
    )
    return f'<s_cert>{inner}</s_cert>'


def load_metadata(jsonl_path: str) -> list:
    with open(jsonl_path, encoding='utf-8') as fh:
        return [json.loads(ln) for ln in fh if ln.strip()]


class CertificateDataset(Dataset):
    def __init__(self, records: list, dataset_root: str):
        self.records = records
        self.root    = Path(dataset_root)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        image = Image.open(self.root / rec['file_name']).convert('RGB')

        pixel_values = processor(image, return_tensors='pt').pixel_values.squeeze()
        target       = fields_to_target(rec)

        input_ids = processor.tokenizer(
            target,
            add_special_tokens=False,
            max_length=CERT_MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        ).input_ids.squeeze()

        labels = input_ids.clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}


print('Loading dataset splits ...')
train_records = load_metadata(f'{DATASET_PATH}/metadata_train.jsonl')
val_records   = load_metadata(f'{DATASET_PATH}/metadata_val.jsonl')
test_records  = load_metadata(f'{DATASET_PATH}/metadata_test.jsonl')

train_ds = CertificateDataset(train_records, DATASET_PATH)
val_ds   = CertificateDataset(val_records,   DATASET_PATH)
test_ds  = CertificateDataset(test_records,  DATASET_PATH)

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}')
print('\nSample target string:')
print(fields_to_target(train_records[0]))

In [ ]:
import os, torch
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Allocator: reduces fragmentation-related OOM
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Gradient checkpointing: trades ~20% speed for ~35% less VRAM
model.gradient_checkpointing_enable()

# Clear any leftover allocations before the trainer starts
torch.cuda.empty_cache()


def collate_fn(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'labels':       torch.stack([b['labels']       for b in batch]),
    }


training_args = Seq2SeqTrainingArguments(
    output_dir=CKPT_PATH,
    # -- Volume ------------------------------------------------------------------
    num_train_epochs=15,
    # -- Batch: bs=1, accum=16 -> effective batch 16 (same as before) -----------
    # Donut's Swin-B encoder is very deep; bs=2 exhausts the T4's 15 GB VRAM.
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    per_device_eval_batch_size=1,
    # -- Optimiser ---------------------------------------------------------------
    # Adafactor uses ~3x less memory than AdamW (no first/second moment vectors).
    optim='adafactor',
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    # -- Mixed precision + memory ------------------------------------------------
    fp16=True,
    gradient_checkpointing=True,
    # -- Logging & evaluation ----------------------------------------------------
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=500,
    # -- Checkpointing (Drive survives Colab session resets) ---------------------
    save_strategy='steps',
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    # -- Misc --------------------------------------------------------------------
    predict_with_generate=False,
    report_to='none',
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

print('Starting fine-tuning...')
print(f'Steps per epoch (eff.): {len(train_ds) // 16}')
print(f'Total epochs           : {training_args.num_train_epochs}')
print(f'Effective batch size   : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'Optimiser              : {training_args.optim}')
print(f'Checkpoints            -> {CKPT_PATH}\n')

trainer.train()

best_ckpt = f'{CKPT_PATH}/best_model'
trainer.save_model(best_ckpt)
processor.save_pretrained(best_ckpt)
print(f'\nBest model saved to {best_ckpt}')

## 📊 Section 4 — Evaluation

Metrics:
- **Field-level exact match** (all 7 fields, case-insensitive)
- **Character Error Rate (CER)** for free-text fields
- **Confusion matrix** for `pass_class` (closed-set)
- **Qualitative** inspection on test samples


In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
import re, torch

BEST_MODEL_PATH = f'{CKPT_PATH}/best_model'
print(f'Loading best checkpoint from {BEST_MODEL_PATH} ...')
processor = DonutProcessor.from_pretrained(BEST_MODEL_PATH)
model     = VisionEncoderDecoderModel.from_pretrained(BEST_MODEL_PATH)
model.to(DEVICE)
model.eval()
print('✅ Best model loaded.')


def decode_to_fields(token_ids) -> dict:
    """Decode model output token ids → dict of 7 field values."""
    text = processor.tokenizer.decode(token_ids, skip_special_tokens=False)
    result = {}
    for f in FIELD_NAMES:
        m = re.search(rf'<s_{f}>(.*?)</s_{f}>', text, re.DOTALL)
        result[f] = m.group(1).strip() if m else ''
    return result


def run_inference(image: 'PIL.Image.Image') -> dict:
    """Run the fine-tuned model on a single PIL image."""
    pixel_values = processor(image.convert('RGB'), return_tensors='pt').pixel_values.to(DEVICE)
    decoder_start = torch.full(
        (1, 1), model.config.decoder_start_token_id, device=DEVICE
    )
    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_start,
            max_length=CERT_MAX_LENGTH,
            early_stopping=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )
    return decode_to_fields(outputs.sequences[0])

print('decode_to_fields and run_inference helpers defined.')

In [ ]:
import editdistance
from collections import defaultdict
from tqdm import tqdm

FREE_TEXT_FIELDS  = ['student_name', 'university_name', 'course_name', 'authority_name']
EXACT_MATCH_FIELDS = FIELD_NAMES   # all 7

# Evaluate on the test split (or a subset for speed)
EVAL_SUBSET = min(200, len(test_records))  # cap at 200 for speed
eval_records = test_records[:EVAL_SUBSET]

exact_matches = defaultdict(int)
cer_totals    = defaultdict(float)
n             = len(eval_records)

pred_pass_classes = []
true_pass_classes = []

for rec in tqdm(eval_records, desc='Evaluating'):
    img  = Image.open(Path(DATASET_PATH) / rec['file_name']).convert('RGB')
    pred = run_inference(img)

    for f in EXACT_MATCH_FIELDS:
        if pred.get(f, '').strip().lower() == rec.get(f, '').strip().lower():
            exact_matches[f] += 1

    for f in FREE_TEXT_FIELDS:
        p, t = pred.get(f, ''), rec.get(f, '')
        cer = editdistance.eval(p, t) / max(len(t), 1)
        cer_totals[f] += cer

    pred_pass_classes.append(pred.get('pass_class', '').strip())
    true_pass_classes.append(rec.get('pass_class', '').strip())

print(f'\n=== Field-level Exact Match (n={n}) ===')
for f in FIELD_NAMES:
    acc = exact_matches[f] / n * 100
    print(f'  {f:<22}: {acc:5.1f}%')

print(f'\n=== Character Error Rate — free-text fields ===')
for f in FREE_TEXT_FIELDS:
    avg_cer = cer_totals[f] / n
    print(f'  {f:<22}: {avg_cer:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

PASS_CLASSES = ['Distinction', 'First Class', 'Second Class Upper', 'Second Class Lower', 'Pass']

cm   = confusion_matrix(true_pass_classes, pred_pass_classes, labels=PASS_CLASSES)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=PASS_CLASSES)

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=30)
plt.title('pass_class Confusion Matrix')
plt.tight_layout()
plt.show()

pc_acc = sum(p == t for p, t in zip(pred_pass_classes, true_pass_classes)) / len(true_pass_classes)
print(f'pass_class accuracy: {pc_acc*100:.1f}%')

In [ ]:
# Qualitative: show 3 test certificates with predicted vs. ground-truth fields
import matplotlib.pyplot as plt
import random

samples = random.sample(eval_records, min(3, len(eval_records)))

for rec in samples:
    img  = Image.open(Path(DATASET_PATH) / rec['file_name']).convert('RGB')
    pred = run_inference(img)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].imshow(img)
    axes[0].set_title(rec['file_name'], fontsize=9)
    axes[0].axis('off')

    report = ''
    for f in FIELD_NAMES:
        ok   = '✅' if pred.get(f,'').strip().lower() == rec.get(f,'').strip().lower() else '❌'
        report += f"{ok} {f}\n  GT  : {rec.get(f,'')}\n  Pred: {pred.get(f,'')}\n\n"

    axes[1].text(0.02, 0.98, report, transform=axes[1].transAxes,
                 fontsize=9, verticalalignment='top', fontfamily='monospace')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

## 🔍 Section 5 — Local Inference Validation

Upload any certificate image (real or synthetic) and verify that the model
always returns a valid JSON with **all 7 keys** — even for fields it cannot
read (returned as empty string `""`).


In [ ]:
from google.colab import files
from PIL import Image
import io, json

print('Upload a certificate image (JPG or PNG):')
uploaded = files.upload()

fname = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[fname])).convert('RGB')

print(f'\nImage: {fname}  |  Size: {image.size}')
image.thumbnail((500, 700))

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 8))
plt.imshow(image)
plt.axis('off')
plt.title(fname)
plt.show()

In [ ]:
# Re-open the full-resolution image for inference
full_image = Image.open(io.BytesIO(uploaded[fname])).convert('RGB')

result = run_inference(full_image)

# Guarantee all 7 keys are present (schema contract)
OUTPUT_SCHEMA = {
    'student_name': '',
    'university_name': '',
    'course_name': '',
    'specialization': '',
    'pass_class': '',
    'authority_name': '',
    'issue_date': '',
}
output = {**OUTPUT_SCHEMA, **result}   # fill missing keys with ''

print('=== Extracted Fields ===')
print(json.dumps(output, indent=2, ensure_ascii=False))

assert set(output.keys()) == set(OUTPUT_SCHEMA.keys()), \
    '❌ Schema violation — missing keys!'
print('\n✅ Schema valid — all 7 keys present.')